In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import json
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

sys.path.append(os.path.abspath("../py"))
from helpers import load_states 

In [2]:
%load_ext autoreload
%autoreload 2

# ***THIS NOTEBOOK CONTAINS FUNCTIONS FOR ANALYSIS OF SIMULATION ENVIRONMENT AND LEARNING ALGORITHMS*** #

In [3]:
"""
def total_energy(x, v, m, G=6.67430e-11):
    KE = 0.5 * np.sum(m[:, None] * np.sum(v**2, axis=1))
    PE = 0.0
    N = len(m)
    for i in range(N):
        for j in range(i+1, N):
            r = np.linalg.norm(x[i] - x[j])
            if r == 0:
                continue
            else:
                PE -= G * m[i] * m[j] / r
    return KE + PE
"""
def total_energy(x, v, m, G=6.67430e-11):
    x = np.asarray(x)
    v = np.asarray(v)
    m = np.asarray(m)

    if x.ndim == 2:
        # single frame
        KE = 0.5 * np.sum(m * np.sum(v**2, axis=1))

        PE = 0.0
        N = len(m)
        for i in range(N):
            for j in range(i+1, N):
                r = np.linalg.norm(x[i] - x[j])
                if r > 0:
                    PE -= G * m[i] * m[j] / r

        return KE + PE

    elif x.ndim == 3:
        # multiple frames
        KE = 0.5 * np.sum(m[None, :] * np.sum(v**2, axis=2), axis=1)

        PE = np.zeros(x.shape[0])
        N = len(m)
        for i in range(N):
            for j in range(i+1, N):
                r = np.linalg.norm(x[:, i, :] - x[:, j, :], axis=1)
                mask = r > 0
                PE[mask] -= G * m[i] * m[j] / r[mask]

        return KE + PE

    else:
        raise ValueError("x and v must have shape (N,2) or (T,N,2)")

"""
def total_angular_momentum(x, v, m):
    L = np.zeros(3)
    for i in range(len(m)):
        r3 = np.array([x[i,0], x[i,1], 0])
        v3 = np.array([v[i,0], v[i,1], 0])
        L += m[i] * np.cross(r3, v3)
    return np.linalg.norm(L)
"""

def total_angular_momentum(x, v, m):
    # barycentric adjustment
    x = np.asarray(x)
    v = np.asarray(v)
    m = np.asarray(m)

    M = np.sum(m)

    if x.ndim == 2:
        # single frame
        r_cm = np.sum(m[:, None] * x, axis=0) / M
        v_cm = np.sum(m[:, None] * v, axis=0) / M

        L = np.zeros(3)
        for i in range(len(m)):
            r3 = np.array([x[i,0] - r_cm[0], x[i,1] - r_cm[1], 0.0])
            v3 = np.array([v[i,0] - v_cm[0], v[i,1] - v_cm[1], 0.0])
            L += m[i] * np.cross(r3, v3)

        return np.linalg.norm(L)

    elif x.ndim == 3:
        # multiple frames
        r_cm = np.sum(m[None, :, None] * x, axis=1) / M
        v_cm = np.sum(m[None, :, None] * v, axis=1) / M

        L = np.zeros((x.shape[0], 3))
        for i in range(len(m)):
            r3 = np.zeros((x.shape[0], 3))
            v3 = np.zeros((x.shape[0], 3))

            r3[:, :2] = x[:, i, :] - r_cm
            v3[:, :2] = v[:, i, :] - v_cm

            L += m[i] * np.cross(r3, v3)

        return np.linalg.norm(L, axis=1)

    else:
        raise ValueError("x and v must have shape (N,2) or (T,N,2)")

"""
def total_linear_momentum(v, m): 
    P = np.sum(m[:, None] * v, axis=0) 
    return np.linalg.norm(P)
"""

def total_linear_momentum(v, m):
    v = np.asarray(v)
    m = np.asarray(m)

    if v.ndim == 2:
        # single frame: (N, D)
        P = np.sum(m[:, None] * v, axis=0)
        return np.linalg.norm(P)

    elif v.ndim == 3:
        # multiple frames: (T, N, D)
        P = np.sum(m[None, :, None] * v, axis=1)
        return np.linalg.norm(P, axis=1)

    else:
        raise ValueError("v must have shape (N,D) or (T,N,D)")

"""
def center_of_mass(x, m):
    return np.linalg.norm(np.sum(m[:,None] * x, axis=0) / np.sum(m))
"""

def center_of_mass(x, m):
    x = np.asarray(x)
    m = np.asarray(m)

    if x.ndim == 2:
        r_cm = np.sum(m[:, None] * x, axis=0) / np.sum(m)
        return np.linalg.norm(r_cm)

    elif x.ndim == 3:
        r_cm = np.sum(m[None, :, None] * x, axis=1) / np.sum(m)
        return np.linalg.norm(r_cm, axis=1)

    else:
        raise ValueError("x must have shape (N,2) or (T,N,2)")



In [4]:
def numerical_analysis(states, title="Numerical Analysis", save_path=None):
    """
    Combined numerical conservation analysis:
        - Relative energy error
        - Relative angular momentum drift
        - Relative linear momentum drift
        - Center of mass drift magnitude
    """

    t = np.array([s["t"] for s in states])

    # -------------------------
    # Energy
    # -------------------------
    E0 = total_energy(states[0]["x"], states[0]["v"], states[0]["m"])
    energy_error = np.array([
        (total_energy(s["x"], s["v"], s["m"]) - E0) /
        (abs(E0) if abs(E0) != 0 else 1)
        for s in states
    ])

    # -------------------------
    # Angular Momentum
    # -------------------------
    L0 = total_angular_momentum(states[0]["x"], states[0]["v"], states[0]["m"])
    dL = np.array([
        (total_angular_momentum(s["x"], s["v"], s["m"]) - L0) /
        (abs(L0) if abs(L0) != 0 else 1)
        for s in states
    ])

    # -------------------------
    # Linear Momentum
    # -------------------------
    M0 = total_linear_momentum(states[0]["v"], states[0]["m"])
    dM = np.array([
        (total_linear_momentum(s["v"], s["m"]) - M0) /
        (abs(M0) if abs(M0) != 0 else 1)
        for s in states
    ])

    # -------------------------
    # Center of Mass Drift
    # -------------------------
    com = np.array([
        np.linalg.norm(center_of_mass(s["x"], s["m"]))
        for s in states
    ])

    # =========================
    # Plot
    # =========================
    fig, axes = plt.subplots(4, 1, figsize=(12, 14), sharex=True)

    axes[0].plot(t, energy_error)
    axes[0].set_ylabel("Energy Difference [J]")
    axes[0].set_title("Relative Energy Error")

    axes[1].plot(t, dL)
    axes[1].set_ylabel("Angular Momentum Difference [kg\u00B7m\u00B2/s]")
    axes[1].set_title("Relative Angular Momentum Drift")

    axes[2].plot(t, dM)
    axes[2].set_ylabel("Linear Momentum Difference [kg\u00B7m/s]")
    axes[2].set_title("Relative Linear Momentum Drift")

    axes[3].plot(t, com)
    axes[3].set_ylabel("Distance from origin [m]")
    axes[3].set_xlabel("Time [s]")
    axes[3].set_title("Center of Mass Drift")

    for ax in axes:
        ax.grid(alpha=0.3)

    fig.suptitle(title, fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    if save_path is not None:
        plt.savefig(save_path, dpi=300)
    plt.show()

In [5]:
def load_states(path):
    with open(path, "r") as f:
        states = json.load(f)

    # Optional: convert back to NumPy
    for s in states:
        s["x"] = np.array(s["x"])
        s["v"] = np.array(s["v"])
        s["a"] = np.array(s["a"])
        s["m"] = np.array(s["m"])
    return states

In [6]:
def body_characteristics(state, G=6.67430e-11, centre_index=None):
    """
    Compute characteristic values per body for one state.
    Returns a list of dicts, one per body.
    """
    x = state['x']
    v = state['v']
    m = state['m']
    N = len(m)

    # Optional centering
    if centre_index is not None:
        x = x - x[centre_index]
        v = v - v[centre_index]

    chars = []

    for i in range(N):

        # absolute velocity
        speed = np.linalg.norm(v[i])
        # linear momentum
        p_mag =  np.linalg.norm(m[i] * v[i])

        # Angular momentum 
        #L = m[i] * (x[i][0]*v[i][1] - x[i][1]*v[i][0]) #(z-component)
        r3 = np.array([x[i,0], x[i,1], 0])
        v3 = np.array([v[i,0], v[i,1], 0])
        L = m[i] * np.cross(r3, v3)
        L = np.linalg.norm(L)

        kinetic = 0.5 * m[i] * speed**2

        # Potential energy
        potential = 0.0
        for j in range(N):
            if i == j:
                continue
            r = np.linalg.norm(x[j] - x[i])
            potential -= 0.5 * G * m[i] * m[j] / r

        #potential = np.abs(potential)

        chars.append({
            "speed": speed,
            "momentum": p_mag,
            "angular_momentum": L,
            "kinetic_energy": kinetic,
            "potential_energy": potential,
            "total_energy": kinetic + potential
        })

    return chars

In [7]:
def visual_analysis(states, body_names, colors=None, scale=1.0/1.496e11, inspect_body=None, centre_body=None, trail_length=50,
                         xlim=(-10,10), ylim=(-10,10), interval=50, save_path=None):
    """
    Visualize a precomputed trajectory of bodies with center on the Sun (0,0).

    states : list of dict
        Each dict must contain 'x' (Nx2 array) and 't' (float).
    body_names : list of str
        Names of bodies (length N).
    colors : list of str 
        Colors for bodies.
    scale : float
        Scaling factor for positions.
    centre_body : str
        name of the body from body_names list to centre the visualisation around
    trail_length : int
        How many past positions to show in trail.
    xlim, ylim : tuple
        Plot limits.
    interval : int
        Milliseconds between frames.
    save_path : str or None
        If given, saves animation to file (requires ffmpeg).
    """
    colors = colors
    n_bodies = len(body_names)

    if((inspect_body is not None) and (inspect_body in body_names)):
        inspect_index = np.where(body_names == inspect_body)[0][0]
    else:
        inspect_index = None
    
    if((centre_body is not None) and (centre_body in body_names)):
        centre_body_index = np.where(body_names == centre_body)[0][0]
    else:
        centre_body_index = None

    # TODO: states preproc: scale and offset calc
    
    # Figure setup
    fig, ax = plt.subplots(figsize=(12,12))
    ax.set_aspect("equal")
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("x (scaled)")
    ax.set_ylabel("y (scaled)")
    ax.set_title("Sol System")
    
    #Scatter objects for bodies positions
    scatters = [ax.scatter([], [], color=colors[i], s=80, label=body_names[i]) for i in range(n_bodies)]

    # Time text
    time_text = ax.text(0.02, 0.98, "", transform=ax.transAxes, ha="left", va="top", fontsize=9, bbox=dict(facecolor="white", alpha=0.6, edgecolor="none"))

    # Body info text
    info_text = ax.text(0.02, 0.02, "", transform=ax.transAxes, ha="left", va="bottom", fontsize=9, bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"))

    
    def init():    
        if(centre_body_index is not None):
            centre_body_offset = states[0]['x'][centre_body_index]
        else:
            centre_body_offset = 0
            
        positions = (states[0]['x'] - centre_body_offset) * scale
        i = 0
        for sc in scatters:
            sc.set_offsets([positions[i]])
            i += 1
            
        time_text.set_text(f"t = {states[0]['t']:.1f} s")

        state = states[0]
        chars = body_characteristics(state, centre_index=centre_body_index)
        if inspect_index:
            c = chars[inspect_index]
        else: 
            c = chars[centre_body_index]

        info_text.set_text(
            f"{inspect_body}\n"
            f"|v| = {c['speed']:.3e} m/s\n"
            f"|p| = {c['momentum']:.3e} kg·m/s\n"
            f"L = {c['angular_momentum']:.3e} kg·m²/s\n"
            f"T = {c['kinetic_energy']:.3e} J\n"
            f"U = {c['potential_energy']:.3e} J\n"
            f"E = {c['total_energy']:.3e} J"
        )

        return tuple(scatters) + (time_text, info_text)

    
    def update(frame):
        state = states[frame]
        
        if(centre_body_index is not None):
            centre_body_offset = state['x'][centre_body_index]
        else:
            centre_body_offset = 0
        
        positions = (state['x'] - centre_body_offset) * scale
        t = state['t']

        # Update bodies positions
        for i, name in enumerate(body_names):
            scatters[i].set_offsets([positions[i]])

        # Update time
        time_text.set_text(f"t = {t:.1f} s")

        # Update info 
        chars = body_characteristics(state, centre_index=centre_body_index)
        if inspect_index:
            c = chars[inspect_index]
        else: 
            c = chars[centre_body_index]

        info_text.set_text(
            f"{inspect_body}\n"
            f"|v| = {c['speed']:.3e} m/s\n"
            f"|p| = {c['momentum']:.3e} kg·m/s\n"
            f"L = {c['angular_momentum']:.3e} kg·m²/s\n"
            f"T = {c['kinetic_energy']:.3e} J\n"
            f"U = {c['potential_energy']:.3e} J\n"
            f"E = {c['total_energy']:.3e} J"
        )

        return tuple(scatters) + (time_text, info_text)

    
    anim = FuncAnimation(fig, update, frames=len(states), init_func=init, blit=False, interval=interval)

    plt.legend()

    if save_path is not None:
        anim.save(save_path, writer="pillow")

    return anim

In [8]:
def plot_final_return_distribution(root_folder, save_path=None):
    """
    Expects structure:
    root_folder/
        algorithm_1/
            run_x/metadata.json
        algorithm_2/
            run_y/metadata.json
    """

    algo_results = {}

    # Loop algorithms
    for algo in os.listdir(root_folder):
        algo_path = os.path.join(root_folder, algo)

        if not os.path.isdir(algo_path):
            continue

        final_returns = []

        # Loop runs
        for run in os.listdir(algo_path):
            run_path = os.path.join(algo_path, run)
            metadata_path = os.path.join(run_path, "metadata.json")

            if not os.path.isfile(metadata_path):
                continue

            with open(metadata_path, "r") as f:
                data = json.load(f)

            if len(data) == 0:
                continue

            # Final episode reward
            final_reward = data[-1]["reward"]
            final_returns.append(final_reward)

        if final_returns:
            algo_results[algo] = final_returns

    # Plot
    plt.figure(figsize=(8, 5))
    plt.boxplot(algo_results.values())
    plt.xticks(range(1, len(algo_results) + 1), ["Advantage Actor-Critic", "Monte Carlo REINFORCE"])#algo_results.keys())
    plt.ylabel("Final Episode Return")
    plt.title("Distribution of Final Returns Across Runs")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

In [9]:
def analyze_multiple_runs(parent_folder, max_episodes, training_name="Multi-Run Training Summary", ma_window=10, save_path=None):
    """
    Analyze multiple RL runs stored in subfolders of parent_folder.

    Each subfolder must contain a file named 'metadata.json'.
    """
    
    run_rewards = []
    run_success_rates = []
    run_convergence_episodes = []
    run_convergence_times = []

    # -----------------------------
    # Load runs
    # -----------------------------
    for run_folder in sorted(os.listdir(parent_folder)):
        run_path = os.path.join(parent_folder, run_folder)

        if not os.path.isdir(run_path):
            continue

        log_path = os.path.join(run_path, "metadata.json")
        if not os.path.isfile(log_path):
            continue

        with open(log_path, "r") as f:
            data = json.load(f)

        rewards = np.array([d["reward"] for d in data])
        successes = np.array([d["success"] for d in data])
        times = np.array([d["time"] for d in data])

        total_episodes = len(rewards)
        if total_episodes == 0:
            continue

        # Success rate
        success_rate = np.sum(successes) / total_episodes * 100
        run_success_rates.append(success_rate)

        # Convergence
        converged = total_episodes < max_episodes
        if converged:
            run_convergence_episodes.append(total_episodes)
            run_convergence_times.append(np.sum(times))
        else:
            run_convergence_episodes.append(None)
            run_convergence_times.append(None)

        # Moving average
        if len(rewards) >= ma_window:
            ma_rewards = np.convolve(
                rewards,
                np.ones(ma_window) / ma_window,
                mode="valid"
            )
        else:
            ma_rewards = rewards

        run_rewards.append(ma_rewards)

    if len(run_rewards) == 0:
        raise ValueError("No valid metadata.json files found.")

    # -----------------------------
    # Pad to max_episodes
    # -----------------------------
    padded_rewards = []

    for r in run_rewards:
        padded = np.full(max_episodes, np.nan)
        length = min(len(r), max_episodes)
        padded[:length] = r[:length]
        padded_rewards.append(padded)

    padded_rewards = np.array(padded_rewards)

    # Safe statistics
    valid_counts = np.sum(~np.isnan(padded_rewards), axis=0)
    valid_mask = valid_counts > 0

    mean_rewards = np.full(max_episodes, np.nan)
    std_rewards = np.full(max_episodes, np.nan)

    mean_rewards[valid_mask] = np.nanmean(
        padded_rewards[:, valid_mask], axis=0
    )
    std_rewards[valid_mask] = np.nanstd(
        padded_rewards[:, valid_mask], axis=0
    )

    episodes = np.arange(1, max_episodes + 1)

    # -----------------------------
    # Summary statistics
    # -----------------------------
    mean_success = np.mean(run_success_rates)
    std_success = np.std(run_success_rates)

    conv_eps = [e for e in run_convergence_episodes if e is not None]
    conv_times = [t for t in run_convergence_times if t is not None]

    mean_conv_ep = np.mean(conv_eps) if conv_eps else None
    mean_conv_time = np.mean(conv_times) if conv_times else None

    # -----------------------------
    # Plot
    # -----------------------------
    fig, ax = plt.subplots(figsize=(16, 6))

    # Individual runs
    for r in padded_rewards:
        ax.plot(episodes, r, alpha=0.2)

    # Mean curve
    ax.plot(episodes, mean_rewards, linewidth=2, label="Mean Reward")

    # Std band
    ax.fill_between(
        episodes[valid_mask],
        (mean_rewards - std_rewards)[valid_mask],
        (mean_rewards + std_rewards)[valid_mask],
        alpha=0.2,
        label="Standard Deviation"
    )

    ax.set_xlim(1, max_episodes)
    ax.set_xlabel("Episode")
    ax.set_ylabel("Reward")
    ax.set_title(f"{training_name}")
    ax.grid(True)

    # Legend INSIDE plot
    ax.legend(loc="upper left")

    # -----------------------------
    # Summary box OUTSIDE plot
    # -----------------------------
    summary_text = (
        f"Runs: {len(run_rewards)}\n"
        f"Mean Success: {mean_success:.2f}% \u00B1 {std_success:.2f}\n"
    )

    if mean_conv_ep is not None:
        summary_text += (
            f"Mean Conv. Episode: {mean_conv_ep:.1f}\n"
            f"Mean Conv. Time: {mean_conv_time:.2f}s"
        )
    else:
        summary_text += "No Convergence Achieved"

    # Add text in figure coordinates (outside axes)
    #fig.text(0.86, 0.75, summary_text, verticalalignment="top", bbox=dict(boxstyle="round", alpha=0.1))

    plt.tight_layout(rect=[0, 0, 0.85, 1])

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

    return {
        "num_runs": len(run_rewards),
        "mean_success_rate": mean_success,
        "std_success_rate": std_success,
        "mean_convergence_episode": mean_conv_ep,
        "mean_convergence_time_sec": mean_conv_time,
    }

In [10]:
def analyze_rl_training(path, max_episodes, training_name="Training Summary", ma_window=10, save_path=None):
    """
    Analyze a single RL training run from JSON log.

    Parameters
    ----------
    path : str
        Path to JSON log file.
    max_episodes : int
        Maximum configured training episodes.
    training_name : str
        Custom name shown in plot title.
    ma_window : int
        Moving average window size.
    save_path : str or None
        If provided, saves plot to this path.
    """

    # -------------------------
    # Load data
    # -------------------------
    with open(path, "r") as f:
        data = json.load(f)

    episodes = np.array([d["episode"] for d in data])
    rewards = np.array([d["reward"] for d in data])
    successes = np.array([d["success"] for d in data])
    times = np.array([d["time"] for d in data])

    total_episodes = len(episodes)

    # -------------------------
    # Success rate
    # -------------------------
    success_rate = np.sum(successes) / total_episodes * 100

    # -------------------------
    # Moving average
    # -------------------------
    if len(rewards) >= ma_window:
        ma_rewards = np.convolve(
            rewards,
            np.ones(ma_window) / ma_window,
            mode="valid"
        )
        ma_episodes = episodes[ma_window - 1:]
    else:
        ma_rewards = rewards
        ma_episodes = episodes

    # -------------------------
    # Convergence (from training logic)
    # -------------------------
    converged = total_episodes < max_episodes

    if converged:
        convergence_episode = episodes[-1]
        convergence_time = np.sum(times)
    else:
        convergence_episode = None
        convergence_time = None

    # -------------------------
    # Plot
    # -------------------------
    plt.figure(figsize=(11, 6))

    plt.plot(episodes, rewards, alpha=0.3, label="Reward")
    plt.plot(ma_episodes, ma_rewards, label="Moving Avg")

    if converged:
        plt.axvline(
            x=convergence_episode,
            linestyle="--",
            label="Convergence"
        )

    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.title(f"{training_name} - Training Reward Curve")
    plt.legend()
    plt.grid(True)

    # -------------------------
    # Training summary on plot
    # -------------------------
    summary_text = (
        f"Episodes: {total_episodes}\n"
        f"Success Rate: {success_rate:.2f}%\n"
        f"Converged: {converged}"
    )

    if converged:
        summary_text += (
            f"\nConv. Episode: {convergence_episode}"
            f"\nTime to Conv.: {convergence_time:.2f}s"
        )

    plt.gca().text(
        0.02,
        0.98,
        summary_text,
        transform=plt.gca().transAxes,
        verticalalignment="top",
        bbox=dict(boxstyle="round", alpha=0.1)
    )

    # -------------------------
    # Save if requested
    # -------------------------
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

    # -------------------------
    # Return structured results
    # -------------------------
    return {
        "training_name": training_name,
        "total_episodes": total_episodes,
        "success_rate_percent": success_rate,
        "converged": converged,
        "convergence_episode": convergence_episode,
        "convergence_time_sec": convergence_time,
    }

In [11]:
def plot_actions_for_episode(run_folder, episode_number, zero_indexed=False, save_path=None):
    """
    run_folder: path containing metadata.json + actions.json
    episode_number: episode to visualize
    zero_indexed: True if episode numbering starts at 0
    """

    metadata_path = os.path.join(run_folder, "metadata.json")
    actions_path = os.path.join(run_folder, "actions.json")

    # Load actions
    with open(actions_path, "r") as f:
        actions_all = json.load(f)

    # Convert episode number to index
    if zero_indexed:
        episode_idx = episode_number
    else:
        episode_idx = episode_number - 1

    if episode_idx >= len(actions_all):
        print("Episode index out of range.")
        return

    actions = np.array(actions_all[episode_idx])
    timesteps = np.arange(len(actions))

    # Plot
    plt.figure(figsize=(14, 4))
    plt.step(timesteps, actions, where="post")

    plt.yticks(
        [0, 1, 2, 3, 4],
        ["No thrust", r"$+x$ thrust", r"$-x$ thrust", r"$+y$ thrust", r"$-y$ thrust"]
    )

    plt.xlabel("Timestep")
    plt.ylabel("Action")
    plt.title(f"Actions over time, episode {episode_number}")
    plt.grid(True, axis="x", alpha=0.2)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        
    plt.show()
